# DeepGuard — WildDeepfake test pilot

One-click pilot using the public Hugging Face mirror of WildDeepfake. The official project describes WildDeepfake as 7,314 face sequences from 707 internet-collected deepfake videos; its repository now points users to Hugging Face for download.

This notebook downloads only four small test archives (~260 MB total), not the full ~10 GB test set or 72.8 GB mirror. It stores the archives on Google Drive and extracts them locally for a quick integrity/label test.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, tarfile, json, shutil
drive.mount('/content/drive', force_remount=False)
ROOT=Path('/content/drive/MyDrive/DeepGuard')
ARCH=ROOT/'datasets/WildDeepfake/pilot_archives'
OUT=ROOT/'datasets/WildDeepfake/pilot_extracted'
ARCH.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
print('Drive destination:',ARCH)
print('Free Drive GiB:',round(shutil.disk_usage('/content/drive').free/1024**3,1))


In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import hf_hub_download
repo='xingjunm/WildDeepfake'
files=[
  ('deepfake_in_the_wild/fake_test/1.tar.gz','fake_1.tar.gz'),
  ('deepfake_in_the_wild/fake_test/10.tar.gz','fake_10.tar.gz'),
  ('deepfake_in_the_wild/real_test/104.tar.gz','real_104.tar.gz'),
  ('deepfake_in_the_wild/real_test/12.tar.gz','real_12.tar.gz'),
]
manifest=[]
for remote,name in files:
    target=ARCH/name
    if target.exists() and target.stat().st_size>0:
        print('Already present:',name)
        p=target
    else:
        print('Downloading:',remote)
        cached=hf_hub_download(repo_id=repo,filename=remote,repo_type='dataset',local_dir=str(ARCH),local_dir_use_symlinks=False)
        p=Path(cached)
        # hf_hub_download preserves nested path; move to a stable flat name
        if p != target: shutil.move(str(p),str(target))
    manifest.append({'remote':remote,'local':str(target),'bytes':target.stat().st_size})
(ARCH/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Downloaded files:',len(manifest))
print('Total GiB:',round(sum(x['bytes'] for x in manifest)/1024**3,3))


In [ ]:
# Extract only the four pilot archives into Drive.
for p in sorted(ARCH.glob('*.tar.gz')):
    dest=OUT/p.stem.replace('.tar','')
    dest.mkdir(parents=True,exist_ok=True)
    print('Extracting',p.name)
    with tarfile.open(p,'r:gz') as tf:
        tf.extractall(dest)
print('Extraction complete:',OUT)


In [ ]:
# Inventory: count images and verify fake/real separation.
from collections import Counter
counts=Counter()
for p in OUT.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'}:
        counts['images']+=1
        counts['bytes']+=p.stat().st_size
print(dict(counts))
for folder in sorted(OUT.iterdir()):
    if folder.is_dir():
        n=sum(1 for p in folder.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'})
        print(folder.name, n, 'images')
assert counts['images']>0, 'No extracted images found'
print('PILOT OK — WildDeepfake test data is readable.')


In [ ]:
# Create a compact manifest for DeepGuard. Labels are determined by the source test folder.
import pandas as pd
rows=[]
for p in OUT.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg'}:
        s=str(p).lower()
        label=1 if 'fake_' in s else 0 if 'real_' in s else None
        if label is not None: rows.append({'path':str(p),'label':label})
df=pd.DataFrame(rows)
df.to_csv(ROOT/'datasets/WildDeepfake/pilot_manifest.csv',index=False)
print(df.label.value_counts().sort_index())
print('Manifest:',ROOT/'datasets/WildDeepfake/pilot_manifest.csv')


## Next step

If this pilot completes, we can plug the extracted face sequences into the DeepGuard feature extractor and then Xception. We will keep this test subset separate from LR calibration so it remains a small external validation pilot.